# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. Usaremos Wine Quality como caso didáctico de clasificación
multiclase: predeciremos la calidad original y no debe interpretarse como garantía comercial.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos siete runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.joblib` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==2.2.3" "matplotlib==3.11.1" "joblib==1.5.3" "xgboost==3.0.5"

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import joblib
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import re
import os

STUDENT_ALIAS = "202637883@alu.comillas.edu"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE - 1.0) < 1e-9
MIN_VALIDATION_F1_MACRO = 0.28
QUALITY_MIN = 3
QUALITY_MAX = 8
NUM_CLASSES = QUALITY_MAX - QUALITY_MIN + 1
# Esta ruta es relativa a semana1/modules/01-mlflow-databricks-foundations/notebooks/.
DATASET_PATH = "../../../data/raw/WineQT.csv"

regex = re.compile(
    r"[A-Za-z0-9._%+-]+@alu\.comillas\.edu",
    re.IGNORECASE,
)

# TODO 1: valida el alias con una expresión regular y configura el tracking de Databricks.
if regex.fullmatch(STUDENT_ALIAS.strip()) is None:
    raise NotImplementedError("El alias no es válido")

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")


## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío, falta el target o `Id` se cuela como predictor. Construye un `quality_report`
serializable como JSON y confirma que el target contiene al menos dos clases.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "quality"
EXPECTED_FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

quality_report = {}

error = False
target = "quality"

datalen = quality_report['Numero de filas'] = len(data)
datacolumns = quality_report['Columnas recibidas'] = data.columns.tolist()
target_values = data["quality"].unique().tolist()

if datalen == 0:
    error = True
if "quality" not in data.columns:
    error = True
if not set(EXPECTED_FEATURES).issubset(set(datacolumns)):
    error = True
if "quality" in data.columns:
    if len(target_values) < 2:
        error = True
    quality_report['Variable objetivo'] = {"quality": target_values}
else:
    quality_report['Variable objetivo'] = "No disponible"


if error:
    raise ValueError("Contrato incumplido. Reportes en quality_report")

X = data[EXPECTED_FEATURES]
y = data[TARGET]

print(quality_report)

# TODO 2: valida el contrato y crea X, y y quality_report.
# Hint: excluye `Id` y conserva quality como problema multiclase. XGBoost requiere etiquetas
# consecutivas desde 0: valida las calidades 3..8 y crea quality_class = quality - QUALITY_MIN.
# Registra filas, duplicados, distribución original/codificada, mapeo de clases, nulos y checks.


## 3. Divide 70 % train, 15 % validación y 15 % test, siempre estratificado

In [0]:
# TODO 3: crea X_train, X_valid, X_test, y_train, y_valid e y_test.
# Hint: aparta primero TEST_SIZE=0.15 con stratify=y. Sobre el 85 % restante, usa
# VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE) = 0.15 / 0.85 para obtener validación.
# Mantén RANDOM_STATE y stratify en las dos llamadas; no uses X_test ni y_test hasta la sección 9.
data["quality_class"] = data["quality"] - QUALITY_MIN

X = data[EXPECTED_FEATURES]
y = data["quality_class"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=VALIDATION_SIZE/(VALIDATION_SIZE+TRAIN_SIZE),
    random_state=RANDOM_STATE,
    stratify=y_temp
)

## 4. Pipeline y funciones de evaluación

In [0]:
def build_pipeline(config: dict) -> Pipeline:
    params = dict(config)                  
    params.pop("candidate_name", None)
    model_type = params.pop("model_type")

    if model_type == "extra_trees":
        estimator = ExtraTreesClassifier(**params)
    elif model_type == "xgboost":
        estimator = XGBClassifier(**params)
    else:
        raise ValueError(f"model_type desconocido. Aceptados: ExtraTreesClassifier o XGBClassifier)")

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator),
    ])

def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    start = time.perf_counter()
    predictions = model.predict(features)
    elapsed = time.perf_counter() - start

    n_rows = max(len(features), 1) 

    return {
        "accuracy":accuracy_score(labels, predictions),
        "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
        "f1_ponderada": f1_score(labels, predictions, average="weighted", zero_division=0),
        "latencia_ms": 1000.0 * elapsed / n_rows,
    }

## 5. Diseña al menos siete candidatos comparables

In [0]:


def extratrees(name, **addedparams):
    params = {
        "candidate_name": name,
        "model_type": "extra_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    params.update(addedparams)
    return params


def xgboost(name, **addedparams):
    params = {
        "candidate_name": name,
        "model_type": "xgboost",
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    params.update(addedparams)
    return params


CANDIDATES = [
    extratrees("et_basic_0"),
    extratrees("et_1", class_weight="balanced"),
    extratrees("et_2", n_estimators=600, min_samples_leaf=3),
    extratrees("et_3", n_estimators=400, max_depth=20, min_samples_leaf=3, class_weight="balanced"),
    xgboost("xgb_1", n_estimators=500, learning_rate=0.05, max_depth=4),
    xgboost("xgb_2", n_estimators=400, learning_rate=0.1, max_depth=6),
    xgboost("xgb_3", n_estimators=800, max_depth=8),
]

assert len(CANDIDATES) >= 7, "Necesitas al menos siete candidatos, incluyendo XGBoost."
assert any(candidate.get("model_type") == "xgboost" for candidate in CANDIDATES)

## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.joblib` y contrato de inferencia.

In [0]:


# No entiendo risk_register ni con qué datos rellenarlo.


dataset_card = {
    "source": "??",
    "purpose": "Ejercicio MLOps, prueba de databricks con MLFlow.",
    "features": EXPECTED_FEATURES,
    "target": "quality_class",
    "target_definition": f"quality original 3..8; quality_class = quality - {QUALITY_MIN}",
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "random_state": RANDOM_STATE},
    "known_limitations": [
        "Desbalance fuerte. ",
        "Pocas muestras",
        "El fallo entre clases está igual de penalizado cuando realmente no debería de estar igual de penalizado entre clases más distantes.",
    ],
}

risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R2", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R3", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R4", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
    ],
}

assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:

train_df = X_train.copy()
train_df["quality_class"] = y_train.values

validation_df = X_val.copy()
validation_df["quality_class"] = y_val.values

train_dataset = mlflow.data.from_pandas(
    train_df,
    source=str(DATASET_PATH),
    targets="quality_class",
    name="wine-quality-train",
)
validation_dataset = mlflow.data.from_pandas(
    validation_df,
    source=str(DATASET_PATH),
    targets="quality_class",
    name="wine-quality-validation",
)

### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, Joblib,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:

XGB_TRUSTED_TYPES = ["numpy.dtype", "xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"]


def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    example = X_val.head(5)
    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        signature=infer_signature(example, model.predict(example)),
        input_example=example,
        skops_trusted_types=XGB_TRUSTED_TYPES,
    )
    mlflow.set_tag("candidate.model_uri", model_info.model_uri)

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "model.joblib")
        joblib.dump(model, path)
        mlflow.log_artifact(path, artifact_path="deployment")
        mlflow.log_metric("joblib_size_bytes", os.path.getsize(path))

    display = ConfusionMatrixDisplay.from_predictions(
        y_val, predictions, labels=list(range(NUM_CLASSES)), colorbar=False
    )
    mlflow.log_figure(display.figure_, f"evaluation/confusion_matrix_{candidate_index}.png")
    plt.close(display.figure_)
    mlflow.log_dict(
        {
            "model_uri": model_info.model_uri,
            "features": list(EXPECTED_FEATURES),
            "output": "quality_class",
            "decode": f"quality = quality_class + {QUALITY_MIN}",
        },
        "deployment/inference_contract.json",
    )
    return model_info.model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:

experiment_id = None
candidate_run_ids = []
LABELS = list(range(NUM_CLASSES))  

for i, config in enumerate(CANDIDATES):
    with mlflow.start_run(run_name=config["candidate_name"]) as run:
        experiment_id = run.info.experiment_id

        mlflow.set_tags({
            "alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "stage": "candidate",
            "candidate.name": config["candidate_name"],
            "candidate.index": i,
            "model_type": config["model_type"],
            "version.xgboost": mlflow.xgboost.get_default_pip_requirements()[0],
            "version.sklearn": mlflow.sklearn.get_default_pip_requirements()[0],
        })
        mlflow.log_params({k: v for k, v in config.items() if k != "candidate_name"})
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(validation_dataset, context="validation")
        mlflow.log_dict(dataset_card, "governance/dataset_card.json")
        mlflow.log_dict(risk_register, "governance/risk_register.json")

        model = build_pipeline(config)
        model.fit(X_train, y_train)
        metrics = evaluate_classifier(model, X_val, y_val)
        mlflow.log_metrics({f"validation.{k}": v for k, v in metrics.items()})

        predictions = model.predict(X_val)
        report = classification_report(
            y_val, predictions, labels=LABELS,
            target_names=[str(l + QUALITY_MIN) for l in LABELS],
            output_dict=True, zero_division=0,
        )
        mlflow.log_dict(report, "evaluation/classification_report.json")

        log_candidate_artifacts(model, predictions, i)
        candidate_run_ids.append(run.info.run_id)

    print(
        f"{i} | {config['candidate_name']:<32} | {config['model_type']:<11} | "
        f"f1_macro={metrics['f1_macro']:.4f} | acc={metrics['accuracy']:.4f} | "
        f"lat={metrics['latencia_ms']:.4f} ms"
    )

assert len(candidate_run_ids) == len(CANDIDATES)

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
filter_string = (
    f"tags.alias = '{STUDENT_ALIAS}' "
    f"AND tags.batch_id = '{BATCH_ID}' "
    f"AND tags.stage = 'candidate'"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=filter_string,
    order_by=["metrics.`validation.f1_macro` DESC"],
)

param_cols = [c for c in runs.columns if c.startswith("params.")]
metric_cols = [c for c in runs.columns if c.startswith("metrics.validation.")]

comparison = runs[["tags.candidate.name", *param_cols, *metric_cols, "run_id"]]

assert len(comparison) == len(CANDIDATES), f"Esperados {len(CANDIDATES)}, encontrados {len(comparison)}"
comparison

## 8. Aplica el gate sin mirar test

Filtra por F1 macro mínimo; ordena por F1 macro DESC, accuracy DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
F1 = "metrics.validation.f1_macro"
ACC = "metrics.validation.accuracy"
LAT = "metrics.validation.latencia_ms"

eligible = runs[runs[F1] >= MIN_VALIDATION_F1_MACRO]

if eligible.empty:
    raise RuntimeError(
        f"Ningún candidato supera validation.f1_macro >= {MIN_VALIDATION_F1_MACRO} "
        f"(mejor: {runs[F1].max():.4f}). No se rebaja el umbral."
    )

best = eligible.sort_values([F1, ACC, LAT], ascending=[False, False, True]).iloc[0]

BEST_RUN_ID = best["run_id"]
BEST_MODEL_URI = best["tags.candidate.model_uri"]

selection_report = {
    "gate": f"validation.f1_macro >= {MIN_VALIDATION_F1_MACRO}",
    "ranking": "f1_macro DESC, accuracy DESC, latencia_ms ASC (validación)",
    "candidates_evaluated": int(len(runs)),
    "candidates_eligible": int(len(eligible)),
    "best_candidate": best["tags.candidate.name"],
    "best_run_id": BEST_RUN_ID,
    "best_model_uri": BEST_MODEL_URI,
    "best_validation_f1_macro": float(best[F1]),
    "test_used_in_selection": False,
    "note": "La selección se hizo solo con métricas de validación; test no participó.",
}

print(json.dumps(selection_report, indent=2, ensure_ascii=False))

## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
F1 = "metrics.validation.f1_macro"
ACC = "metrics.validation.accuracy"
LAT = "metrics.validation.latencia_ms"

eligible = runs[runs[F1] >= MIN_VALIDATION_F1_MACRO]

if eligible.empty:
    raise RuntimeError(
        f"Ningún candidato supera validation.f1_macro >= {MIN_VALIDATION_F1_MACRO}. "
        f"Mejor F1 macro obtenido: {runs[F1].max():.4f}. No se rebaja el umbral."
    )

ranked = eligible.sort_values([F1, ACC, LAT], ascending=[False, False, True])
best = ranked.iloc[0]

BEST_RUN_ID = best["run_id"]
BEST_MODEL_URI = best["tags.candidate.model_uri"]

selection_report = {
    "batch_id": BATCH_ID,
    "alias": STUDENT_ALIAS,
    "gate": f"validation.f1_macro >= {MIN_VALIDATION_F1_MACRO}",
    "ranking": "validation.f1_macro DESC, validation.accuracy DESC, validation.latencia_ms ASC",
    "candidates_evaluated": int(len(runs)),
    "candidates_eligible": int(len(eligible)),
    "best_candidate": best["tags.candidate.name"],
    "best_run_id": BEST_RUN_ID,
    "best_model_uri": BEST_MODEL_URI,
    "best_validation_metrics": {
        "f1_macro": float(best[F1]),
        "accuracy": float(best[ACC]),
        "latencia_ms": float(best[LAT]),
    },
    "test_used_in_selection": False,
    "note": "La selección se hizo solo con métricas de validación; el conjunto de test no participó.",
}

print(json.dumps(selection_report, indent=2, ensure_ascii=False))

## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

catalog, schema = spark.sql("SELECT current_catalog(), current_schema()").first()
safe_alias = re.sub(r"[^a-z0-9_]", "_", STUDENT_ALIAS.lower())
MODEL_NAME = f"{catalog}.{schema}.wine_quality_{safe_alias}"

ref_row = runs[runs["tags.candidate.index"] == "0"].iloc[0]
ref_version = mlflow.register_model(ref_row["tags.candidate.model_uri"], MODEL_NAME).version
best_version = mlflow.register_model(BEST_MODEL_URI, MODEL_NAME).version

client.update_registered_model(
    MODEL_NAME,
    description=f"Clasificador de calidad de vino tinto (6 clases, quality_class = quality - {QUALITY_MIN}). "
                "Pipeline SimpleImputer + modelo. Seleccionado solo con validación.",
)
client.set_registered_model_tag(MODEL_NAME, "alias", STUDENT_ALIAS)
client.set_registered_model_tag(MODEL_NAME, "batch_id", BATCH_ID)

for version, role, row in [(ref_version, "reference", ref_row), (best_version, "winner", best)]:
    client.update_model_version(
        MODEL_NAME, version,
        description=f"{role}: {row['tags.candidate.name']} · "
                    f"validation.f1_macro={float(row['metrics.validation.f1_macro']):.4f}",
    )
    client.set_model_version_tag(MODEL_NAME, version, "role", role)
    client.set_model_version_tag(MODEL_NAME, version, "source_run_id", row["run_id"])

client.set_registered_model_alias(MODEL_NAME, "Champion", ref_version)
client.set_registered_model_alias(MODEL_NAME, "Challenger", best_version)

challenger = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@Challenger")
smoke_preds = np.asarray(challenger.predict(X_val.head(20)))
assert len(smoke_preds) == 20
assert set(smoke_preds.astype(int)) <= set(range(NUM_CLASSES))

client.set_registered_model_alias(MODEL_NAME, "Champion", best_version)
assert client.get_model_version_by_alias(MODEL_NAME, "Champion").version == best_version

client.set_registered_model_alias(MODEL_NAME, "Champion", ref_version)
assert client.get_model_version_by_alias(MODEL_NAME, "Champion").version == ref_version

client.set_registered_model_alias(MODEL_NAME, "Champion", best_version)
champion = client.get_model_version_by_alias(MODEL_NAME, "Champion")
assert champion.version == best_version

print(f"{MODEL_NAME} · Champion = v{champion.version} (referencia v{ref_version})")

## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@Champion")
sample = X_val[EXPECTED_FEATURES].head(3)
champion_preds = np.asarray(champion_model.predict(sample))

assert len(champion_preds) == 3
assert set(champion_preds.astype(int)) <= set(range(NUM_CLASSES))
print(sample.assign(quality_pred=champion_preds.astype(int) + QUALITY_MIN))

## 12. API local con el artefacto Joblib ganador

Resuelve el run de `Champion`, descarga `deployment/model.joblib` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
CHAMPION_RUN_ID = champion.run_id
artifact_dir = tempfile.mkdtemp()

joblib_path = mlflow.artifacts.download_artifacts(
    run_id=CHAMPION_RUN_ID,
    artifact_path="deployment/model.joblib",
    dst_path=artifact_dir,
)

serving_model = joblib.load(joblib_path)
one_row = X_val[EXPECTED_FEATURES].head(1)
one_pred = serving_model.predict(one_row)

assert len(one_pred) == 1 and 0 <= int(one_pred[0]) < NUM_CLASSES
print(Path(joblib_path).name, int(one_pred[0]) + QUALITY_MIN)

### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
class RequestRejected(ValueError):
    pass
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

class PredictionHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def _send_json(self, status: int, payload: dict) -> None:
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _record(self, outcome: str, start: float) -> None:
        with API_STATS_LOCK:
            API_STATS["requests_total"] += 1
            API_STATS[f"requests_{outcome}"] += 1
            API_STATS["latencies_ms"].append(1000.0 * (time.perf_counter() - start))

    def do_GET(self):
        start = time.perf_counter()
        if self.path == "/health":
            self._send_json(200, {"status": "ok", "model_run_id": CHAMPION_RUN_ID})
            self._record("success", start)
        else:
            self._send_json(404, {"error": f"ruta no encontrada: {self.path}"})
            self._record("rejected", start)

    def _parse_instances(self) -> pd.DataFrame:
        length = int(self.headers.get("Content-Length", 0))
        try:
            payload = json.loads(self.rfile.read(length) or b"null")
        except (json.JSONDecodeError, UnicodeDecodeError):
            raise RequestRejected("el cuerpo no es JSON válido")

        if not isinstance(payload, dict):
            raise RequestRejected("el objeto raíz debe ser un objeto JSON")

        instances = payload.get("instances")
        if not isinstance(instances, list) or not instances:
            raise RequestRejected("'instances' debe ser una lista no vacía")

        expected = set(EXPECTED_FEATURES)
        for i, row in enumerate(instances):
            if not isinstance(row, dict):
                raise RequestRejected(f"instances[{i}] debe ser un objeto")
            missing = sorted(expected - row.keys())
            extra = sorted(row.keys() - expected)
            if missing or extra:
                raise RequestRejected(
                    f"instances[{i}] columnas inválidas · ausentes: {missing} · extra: {extra}"
                )

        try:
            return pd.DataFrame(instances, columns=EXPECTED_FEATURES).astype(float)
        except (TypeError, ValueError):
            raise RequestRejected("todas las features deben ser numéricas")

    def do_POST(self):
        start = time.perf_counter()
        if self.path != "/predict":
            self._send_json(404, {"error": f"ruta no encontrada: {self.path}"})
            self._record("rejected", start)
            return

        try:
            frame = self._parse_instances()
            classes = serving_model.predict(frame).astype(int)
            self._send_json(200, {
                "quality_class": classes.tolist(),
                "quality": (classes + QUALITY_MIN).tolist(),
                "model_run_id": CHAMPION_RUN_ID,
            })
            self._record("success", start)
        except RequestRejected as exc:
            self._send_json(400, {"error": str(exc)})
            self._record("rejected", start)
        except Exception as exc:
            self._send_json(500, {"error": f"error interno: {type(exc).__name__}"})
            self._record("error", start)

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:
if "stop_api" in globals():
    stop_api()

api_server = ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
api_server.daemon_threads = True
API_PORT = api_server.server_address[1]
api_thread = threading.Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()
API_STOPPED = False


def stop_api() -> None:
    global API_STOPPED
    if API_STOPPED:
        return
    api_server.shutdown()
    api_server.server_close()
    api_thread.join(timeout=5)
    API_STOPPED = True


print(f"API en http://127.0.0.1:{API_PORT}")

## 13. Pruebas de contrato y de error

In [0]:
def call_json(method: str, path: str, payload=None) -> tuple[int, dict]:
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"http://127.0.0.1:{API_PORT}{path}",
        data=data,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urlopen(request, timeout=10) as response:
            return response.status, json.loads(response.read())
    except HTTPError as exc:
        return exc.code, json.loads(exc.read())


valid_rows = X_val[EXPECTED_FEATURES].head(2).to_dict(orient="records")

try:
    status, body = call_json("GET", "/health")
    assert status == 200 and body["status"] == "ok", (status, body)

    status, body = call_json("POST", "/predict", {"instances": valid_rows})
    assert status == 200 and len(body["quality"]) == len(valid_rows), (status, body)

    status, body = call_json("POST", "/predict", {"instances": [{"alcohol": 10.5}]})
    assert status == 400, (status, body)

    status, body = call_json("POST", "/predict", [1, 2, 3])
    assert status == 400, (status, body)

    print("Pruebas de contrato OK")
finally:
    stop_api()

print(API_STATS["requests_total"], API_STATS["requests_success"], API_STATS["requests_rejected"])

## 14. Registra un run separado para la evidencia del despliegue

In [0]:
with API_STATS_LOCK:
    stats = dict(API_STATS)
    latencies = list(API_STATS["latencies_ms"])

evidence = {
    "model_name": MODEL_NAME,
    "model_version": champion.version,
    "model_alias": "Champion",
    "source_run_id": CHAMPION_RUN_ID,
    "endpoint": "127.0.0.1 (local, puerto automático)",
    "requests_total": stats["requests_total"],
    "requests_success": stats["requests_success"],
    "requests_rejected": stats["requests_rejected"],
    "requests_error": stats["requests_error"],
    "latency_ms_p50": float(np.percentile(latencies, 50)) if latencies else None,
    "latency_ms_p95": float(np.percentile(latencies, 95)) if latencies else None,
    "latency_ms_max": float(max(latencies)) if latencies else None,
    "payloads_logged": False,
}

with mlflow.start_run(run_name="deployment-evidence") as deployment_run:
    mlflow.set_tags({
        "alias": STUDENT_ALIAS,
        "batch_id": BATCH_ID,
        "stage": "deployment",
        "model.name": MODEL_NAME,
        "model.version": champion.version,
        "model.alias": "Champion",
        "source_run_id": CHAMPION_RUN_ID,
    })
    mlflow.log_metrics({
        "deployment.requests_total": evidence["requests_total"],
        "deployment.requests_success": evidence["requests_success"],
        "deployment.requests_rejected": evidence["requests_rejected"],
        "deployment.requests_error": evidence["requests_error"],
        **{f"deployment.{k}": v for k, v in evidence.items() if k.startswith("latency_ms_") and v is not None},
    })
    mlflow.log_dict(evidence, "deployment/evidence.json")
    DEPLOYMENT_RUN_ID = deployment_run.info.run_id

print(DEPLOYMENT_RUN_ID)
print(json.dumps(evidence, indent=2, ensure_ascii=False))

## 15. Apaga la API y entrega el resumen del ciclo

In [0]:
if not API_STOPPED:
    api_server.shutdown()
    api_server.server_close()
    api_thread.join(timeout=5)
    API_STOPPED = True

assert not api_thread.is_alive(), "El hilo del servidor sigue vivo"
assert api_server.socket.fileno() == -1, "El socket del servidor sigue abierto"

summary = {
    "experiment_id": experiment_id,
    "batch_id": BATCH_ID,
    "alias": STUDENT_ALIAS,
    "winner": best["tags.candidate.name"],
    "winner_run_id": BEST_RUN_ID,
    "winner_validation_f1_macro": float(best["metrics.validation.f1_macro"]),
    "model_name": MODEL_NAME,
    "model_version": champion.version,
    "model_alias": "Champion",
    "deployment_run_id": DEPLOYMENT_RUN_ID,
    "test_used_in_selection": False,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))